In [ ]:
import pandas as pd
import numpy as np
import math
from scipy import stats as scipy_stats
from scipy.optimize import linprog
from typing import Union, List, Tuple, Callable

In [ ]:
def bin_target(df: pd.DataFrame, target_col: str) -> dict:
    """
    Discretise a target column using Scott's rule and return a mapping
    of bin_id -> list of row indices.

    Scott's rule:
        bin_width  h = 3.5 * std(x) * n^(-1/3)
        n_bins       = ceil((max(x) - min(x)) / h)

    Parameters
    ----------
    df         : input DataFrame
    target_col : name of the column to discretise

    Returns
    -------
    dict  {bin_id (int, 0-based): [row_index, ...]}
    """
    x = df[target_col].dropna().values
    n = len(x)

    if n < 2:
        raise ValueError("Need at least 2 non-null values to compute bins.")

    std = np.std(x, ddof=1)          # sample standard deviation
    if std == 0:
        raise ValueError("Target column has zero variance; cannot bin.")

    h       = 3.5 * std * (n ** (-1 / 3))          # Scott's bin width
    n_bins  = max(1, math.ceil((x.max() - x.min()) / h))

    # Cut into n_bins equal-width intervals; labels are 0-based integer bin IDs
    bin_series = pd.cut(
        df[target_col],
        bins=n_bins,
        labels=range(n_bins),
        include_lowest=True
    )

    bin_dict = {}
    for bin_id, group in df.groupby(bin_series, observed=True):
        bin_dict[int(bin_id)] = list(group.index)

    return bin_dict, n_bins

In [ ]:
# Sample DataFrame
np.random.seed(42)
df = pd.DataFrame({
    "feature_a": np.random.randn(100),
    "feature_b": np.random.rand(100),
    "target":    np.random.normal(loc=50, scale=15, size=100)
})

bin_map, n_bins = bin_target(df, target_col="target")

print(f"Scott's rule → {n_bins} bins\n")
for bin_id, indices in bin_map.items():
    print(f"  Bin {bin_id:>2d} ({len(indices):>3d} items): {indices[:5]}{'...' if len(indices) > 5 else ''}")

In [ ]:
DIST_MAP = {
    "normal":      scipy_stats.norm,
    "gaussian":    scipy_stats.norm,
    "exponential": scipy_stats.expon,
    "beta":        scipy_stats.beta,
    "gamma":       scipy_stats.gamma,
    "uniform":     scipy_stats.uniform,
    "lognormal":   scipy_stats.lognorm,
}

def make_pdf(
    spec: Union[
        Tuple[str, dict],
        List[Tuple[str, dict, float]]
    ]
) -> Callable:
    """
    Single distribution:
        spec = ("normal", {"loc": 0, "scale": 1})

    Mixture (weights are auto-normalised):
        spec = [
            ("normal",      {"loc": 0, "scale": 1},  0.6),
            ("exponential", {"scale": 2},             0.4),
        ]
    """
    if isinstance(spec, tuple):
        name, params = spec
        dist = DIST_MAP[name.lower()]
        return lambda x: dist.pdf(x, **params)

    total_w    = sum(w for _, _, w in spec)
    components = [(DIST_MAP[n.lower()], p, w / total_w) for n, p, w in spec]

    def mixture_pdf(x):
        return sum(w * d.pdf(x, **p) for d, p, w in components)

    return mixture_pdf

In [ ]:
def least_cardinality_bin(bin_map: dict) -> Tuple[int, int]:
    """Return (bin_id, cardinality) of the bin with the fewest items."""
    bin_id = min(bin_map, key=lambda b: len(bin_map[b]))
    return bin_id, len(bin_map[bin_id])


def _bin_centers(df: pd.DataFrame, target_col: str, n_bins: int) -> np.ndarray:
    """Midpoint of each equal-width bin over the target column range."""
    lo, hi = df[target_col].min(), df[target_col].max()
    edges  = np.linspace(lo, hi, n_bins + 1)
    return (edges[:-1] + edges[1:]) / 2

In [ ]:
EPS = 1e-9   # prevents bins from collapsing to 0 when c = 0

def find_sampling_plan(
    bin_map:       dict,
    df:            pd.DataFrame,
    target_col:    str,
    n_bins:        int,
    pdf:           Callable,
    min_total_pct: float = 0.0,
) -> Tuple[dict, List[int], bool]:
    """
    For c = 0, 1, ..., capacity(trough bin), derive required counts for all
    bins as round((c + eps) * pdf_i / pdf_trough).  The eps ensures that when
    c = 0, non-trough bins round to their minimum non-zero quota rather than
    collapsing to 0.  Collect every feasible level (all bins can supply their
    quota), then pick the one whose total is the smallest value >=
    min_total_pct * len(df).

    Returns
    -------
    required  : {bin_id: required_count} for the chosen level (empty if infeasible)
    peak_bins : bin_ids at the distribution peak
    feasible  : True iff a valid level meeting the total constraint exists
    """
    centers    = _bin_centers(df, target_col, n_bins)
    bin_ids    = sorted(bin_map.keys())
    pdf_vals   = np.array([pdf(centers[i]) for i in bin_ids])

    peak_pdf   = pdf_vals.max()
    trough_pdf = pdf_vals.min()
    peak_bins  = [bid for bid, pv in zip(bin_ids, pdf_vals) if np.isclose(pv, peak_pdf)]
    trough_bid = bin_ids[int(np.argmin(pdf_vals))]

    min_total       = round(min_total_pct * len(df))
    trough_capacity = len(bin_map[trough_bid])

    feasible_levels = []
    for c in range(trough_capacity + 1):
        required = {
            bid: round((c + EPS) * pv / trough_pdf)
            for bid, pv in zip(bin_ids, pdf_vals)
        }
        if all(required[bid] <= len(bin_map[bid]) for bid in bin_ids):
            feasible_levels.append((sum(required.values()), required))

    candidates = [(total, req) for total, req in feasible_levels if total >= min_total]

    if not candidates:
        return {}, peak_bins, False

    _, best_required = min(candidates, key=lambda x: x[0])
    return best_required, peak_bins, True

In [ ]:
def sanity_check(
    bin_map:       dict,
    df:            pd.DataFrame,
    required:      dict,
    min_total_pct: float = 0.0,
) -> pd.DataFrame:
    """
    Compare required vs available cardinality per bin using a pre-computed
    required dict from find_sampling_plan.  Appends a TOTAL row that checks
    the minimum total cardinality constraint (min_total_pct * len(df)).
    """
    min_total = round(min_total_pct * len(df))

    rows = []
    for bid, req in sorted(required.items()):
        available = len(bin_map[bid])
        rows.append({
            "bin_id":     bid,
            "available":  available,
            "required":   req,
            "sufficient": available >= req,
        })

    report    = pd.DataFrame(rows)
    total_req = report["required"].sum()
    total_avl = report["available"].sum()

    report = pd.concat([report, pd.DataFrame([{
        "bin_id":     "TOTAL",
        "available":  total_avl,
        "required":   total_req,
        "sufficient": total_req >= min_total,
    }])], ignore_index=True)

    bins_ok  = report.iloc[:-1]["sufficient"].all()
    total_ok = total_req >= min_total
    status   = "PASSED ✓" if (bins_ok and total_ok) else "FAILED ✗"
    print(f"Sanity check {status}")
    print(f"  Bin capacities : {report.iloc[:-1]['sufficient'].sum()}/{len(report)-1} sufficient")
    print(f"  Total samples  : {total_req} / {len(df)} rows  "
          f"({100*total_req/len(df):.1f}%,  min required: {min_total_pct*100:.0f}% = {min_total}) "
          f"{'✓' if total_ok else '✗'}")
    return report

In [ ]:
np.random.seed(42)
df = pd.DataFrame({
    "feature_a": np.random.randn(500),
    "target":    np.random.normal(loc=50, scale=15, size=500),
})
bin_map, n_bins = bin_target(df, "target")
pdf = make_pdf(("gaussian", {"loc": 50, "scale": 15}))

lc_bin, lc_count = least_cardinality_bin(bin_map)
print(f"Least-cardinality bin: {lc_bin}  (n={lc_count})  |  Total rows: {len(df)}\n")

# case 1: achievable (40% of 500 = 200 rows)
PCT = 0.40
required, peak_bins, ok = find_sampling_plan(bin_map, df, "target", n_bins, pdf, min_total_pct=PCT)
print(f"[min_total_pct={PCT:.0%}]  feasible={ok}  peak_bins={peak_bins}\n")
report = sanity_check(bin_map, df, required, min_total_pct=PCT)
display(report)

# case 2: infeasible (99% of 500 = 495 rows)
print("\n── Infeasible min_total_pct ──")
PCT_BIG = 0.99
_, _, ok2 = find_sampling_plan(bin_map, df, "target", n_bins, pdf, min_total_pct=PCT_BIG)
print(f"[min_total_pct={PCT_BIG:.0%}]  feasible={ok2}")

# case 3: bimodal mixture at 50% coverage
print("\n── Bimodal mixture ──")
pdf_mix = make_pdf([
    ("normal", {"loc": 35, "scale": 8},  0.5),
    ("normal", {"loc": 65, "scale": 8},  0.5),
])
required_mix, peak_bins_mix, ok_mix = find_sampling_plan(
    bin_map, df, "target", n_bins, pdf_mix, min_total_pct=0.50
)
print(f"[min_total_pct=50%]  feasible={ok_mix}  peak_bins={peak_bins_mix}\n")
report_mix = sanity_check(bin_map, df, required_mix, min_total_pct=0.50)
display(report_mix)

In [ ]:
def solve_island_lp(
    order:          np.ndarray,
    bin_ids_sorted: np.ndarray,
    required:       dict,
    eps:            float = 1e-6,
    max_iter:       int = 50,
):
    """
    order          : original df indices sorted ascending by the chosen column (len n)
    bin_ids_sorted : bin_id for each position in `order` (len n, aligned)
    required       : {bin_id: b_k} exact selection count required from each bin

    Minimises  sum_i x_i * (1 - x_{i+1})   [x_{n+1} = 0]
    s.t.       sum_{i in B_k} x_i = b_k,   x_i in {0,1}

    Linearisation: z_i >= x_i - x_{i+1}, z_i >= 0, minimise sum(z_i)
    -- exactly equals x_i(1 - x_{i+1}) for binary x.

    Solved as an LP relaxation (x_i in [0,1]).  At each iteration only the
    single unfixed variable closest to an integer (0 or 1) is fixed (if within
    eps), and the LP is re-solved, until the solution is fully integral.

    Returns (selection: 0/1 array aligned with `order`, mdl: int) or (None, None)
    """
    n = len(order)
    fixed = {}

    for _ in range(max_iter):
        c = np.concatenate([np.zeros(n), np.ones(n)])

        # -z_i + x_i - x_{i+1} <= 0   (x_{n+1} = 0, so last row has no x_{i+1} term)
        rows, b_ub = [], []
        for i in range(n):
            row = np.zeros(2 * n)
            row[i] = 1
            if i + 1 < n:
                row[i + 1] = -1
            row[n + i] = -1
            rows.append(row)
            b_ub.append(0.0)
        A_ub, b_ub = np.array(rows), np.array(b_ub)

        # sum_{i in B_k} x_i = b_k
        eq_rows, b_eq = [], []
        for bin_id, b_k in required.items():
            row = np.zeros(2 * n)
            row[np.where(bin_ids_sorted == bin_id)[0]] = 1
            eq_rows.append(row)
            b_eq.append(b_k)
        A_eq, b_eq = np.array(eq_rows), np.array(b_eq)

        bounds = (
            [(fixed[i], fixed[i]) if i in fixed else (0, 1) for i in range(n)]
            + [(0, None)] * n
        )

        res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                      bounds=bounds, method="highs")
        if not res.success:
            return None, None

        x = res.x[:n]

        if all(abs(v - round(v)) < eps for v in x):
            selection = np.round(x).astype(int)
            mdl = int(sum(
                selection[i] * (1 - (selection[i + 1] if i + 1 < n else 0))
                for i in range(n)
            ))
            return selection, mdl

        # find the unfixed variable closest to an integer (0 or 1)
        best_i, best_dist, best_val = None, None, None
        for i in range(n):
            if i in fixed:
                continue
            dist_to_1 = abs(x[i] - 1)
            dist_to_0 = abs(x[i])
            d, v = (dist_to_1, 1) if dist_to_1 <= dist_to_0 else (dist_to_0, 0)
            if best_dist is None or d < best_dist:
                best_i, best_dist, best_val = i, d, v

        if best_i is not None and best_dist < eps:
            fixed[best_i] = best_val
        else:
            return None, None

    return None, None

In [ ]:
def select_subsets(
    df:        pd.DataFrame,
    bin_map:   dict,
    required:  dict,
    n_subsets: int,
    columns:   List[str] = None,
) -> pd.DataFrame:
    """
    For every candidate column, sort the dataset ascending by that column and
    solve the island-minimisation LP for the row selection that satisfies the
    per-bin quotas (`required`) with minimal MDL (number of contiguous islands
    of selected rows).  Returns the n_subsets simplest (lowest-MDL) results.

    Returns a DataFrame: column | mdl | selected_indices
    """
    if columns is None:
        columns = list(df.columns)

    bin_of_index = {idx: bid for bid, idxs in bin_map.items() for idx in idxs}

    results = []
    for col in columns:
        order = df[col].sort_values(kind="mergesort").index.to_numpy()
        bin_ids_sorted = np.array([bin_of_index[idx] for idx in order])

        selection, mdl = solve_island_lp(order, bin_ids_sorted, required)
        if selection is None:
            continue

        results.append({
            "column":           col,
            "mdl":              mdl,
            "selected_indices": order[selection.astype(bool)].tolist(),
        })

    report = pd.DataFrame(results).sort_values("mdl", kind="mergesort").reset_index(drop=True)
    return report.head(n_subsets)

In [ ]:
N_SUBSETS = 3
subset_report = select_subsets(df, bin_map, required, n_subsets=N_SUBSETS)

print(f"Top {N_SUBSETS} simplest subsets (lowest MDL = fewest islands of selected rows):\n")
for _, row in subset_report.iterrows():
    idxs = row["selected_indices"]
    print(f"  Column '{row['column']}'  →  MDL = {row['mdl']}  "
          f"({len(idxs)} rows selected)  e.g. {idxs[:8]}{'...' if len(idxs) > 8 else ''}")

display(subset_report)